# HW2 Part 2: Single Stock Risk Report

This is a parameterized report for one stock. Change `ticker`, `start_date` and `end_date` in the next cell
(or let papermill do it in `03_run_all_stocks.py`). The data comes from the `ohlc_hw_cleaned` table made in Part 1.
All returns, volatility and drawdown use **adjusted close**.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import scrapbook as sb

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Parameters
ticker = "AAPL"  # Stock ticker to analyze
start_date = "2024-01-01"  # Analysis start date
end_date = "2024-12-31"  # Analysis end date
db_path = "data/data.db"  # Local working copy of the database

In [ ]:
query = """
SELECT ts, open, high, low, close, adj_close, volume
FROM ohlc_hw_cleaned
WHERE ticker = ? AND ts >= ? AND ts <= ?
ORDER BY ts
"""
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date), parse_dates=['ts'], index_col='ts')
conn.close()

if len(df) < 21:
    raise ValueError(f'{ticker}: need at least 21 prices for 20 returns, found {len(df)} between {start_date} and {end_date}.')

print(f'{ticker} from {start_date} to {end_date}: {len(df)} trading days')
print(f'Actual first observation: {df.index.min().date()} | actual last observation: {df.index.max().date()}')
display(df.head())
display(df.tail())

## Risk metrics

- **Daily returns**: log returns of adjusted close
- **Rolling volatility**: 20-day standard deviation of log returns times the square root of 252
- **Drawdown**: how far the price is below its running peak (it can recover, so maximum drawdown is the minimum of this series)
- **Sharpe ratio**: (mean daily return * 252 - 2% risk-free rate) / (std * square root of 252)

In [ ]:
def calculate_returns(prices: pd.Series) -> pd.Series:
    """Calculate daily log returns. The first value is NaN because there is no previous price."""
    return np.log(prices / prices.shift(1))


def calculate_volatility(returns: pd.Series, window: int = 20) -> pd.Series:
    """Calculate annualized rolling volatility (NaN until a full window of returns exists)."""
    return returns.rolling(window).std() * np.sqrt(252)


def calculate_max_drawdown(prices: pd.Series) -> pd.Series:
    """Return the drawdown on every date: (price - running max price) / running max price.

    The values are 0 or negative. The maximum drawdown is drawdown.min().
    """
    running_max = prices.cummax()
    return (prices - running_max) / running_max


def calculate_sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.02) -> float:
    """Calculate the annualized Sharpe ratio. Returns NaN if there are too few returns or zero volatility."""
    returns = returns.dropna()
    if len(returns) < 2:
        return np.nan
    std_return = returns.std()
    if np.isnan(std_return) or std_return == 0:
        return np.nan
    return (returns.mean() * 252 - risk_free_rate) / (std_return * np.sqrt(252))

In [ ]:
df['log_return'] = calculate_returns(df['adj_close'])
df['volatility'] = calculate_volatility(df['log_return'])
df['ma_50'] = df['adj_close'].rolling(50).mean()
df['drawdown'] = calculate_max_drawdown(df['adj_close'])

if len(df) < 50:
    print(f'Only {len(df)} prices: the 50-day moving average needs 50, so it is not shown.')

display(df[['adj_close', 'log_return', 'volatility', 'ma_50', 'drawdown']].tail(10))

## Visualization

Top left: price (line colored by volatility: red above the report's mean rolling volatility, green at or below it, gray while the volatility is not available yet).
Top right: daily returns. Bottom left: rolling volatility. Bottom right: drawdown.

In [ ]:
vol_mean = df['volatility'].mean()
dd_min_date = df['drawdown'].idxmin()
dd_min_value = df['drawdown'].min()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'{ticker} Risk Analysis Report - {start_date} to {end_date}', fontsize=18, fontweight='bold')

# top left: price colored by volatility
ax = axes[0, 0]
for i in range(1, len(df)):
    x_values = df.index[i - 1:i + 1]
    y_values = df['adj_close'].iloc[i - 1:i + 1]
    vol_now = df['volatility'].iloc[i]
    if np.isnan(vol_now):
        line_color = 'gray'
    elif vol_now > vol_mean:
        line_color = 'red'
    else:
        line_color = 'green'
    ax.plot(x_values, y_values, color=line_color, linewidth=2)
ax.plot(df.index, df['ma_50'], color='navy', linestyle='--', linewidth=1.5, label='50-day MA')
ax.plot([], [], color='green', label='Low volatility')
ax.plot([], [], color='red', label='High volatility')
ax.plot([], [], color='gray', label='Volatility not available')
ax.set_title(f'{ticker} Price and 50-day MA')
ax.set_xlabel('Date')
ax.set_ylabel('Adjusted close ($)')
ax.legend(loc='best')
ax.grid(True)

# top right: daily returns
ax = axes[0, 1]
bar_colors = []
for value in df['log_return']:
    if value >= 0:
        bar_colors.append('green')
    else:
        bar_colors.append('red')
ax.bar(df.index, df['log_return'], color=bar_colors, width=1.0)
ax.axhline(0, color='black', linewidth=1)
ax.set_title(f'{ticker} Daily Returns')
ax.set_xlabel('Date')
ax.set_ylabel('Daily log return')
ax.grid(True)

# bottom left: rolling volatility
ax = axes[1, 0]
ax.plot(df.index, df['volatility'], color='purple', linewidth=1.8, label='20-day volatility')
ax.axhline(vol_mean, color='black', linestyle='--', label=f'Mean ({vol_mean:.1%})')
ax.fill_between(df.index, df['volatility'], vol_mean, where=df['volatility'] > vol_mean,
                color='red', alpha=0.3, interpolate=True, label='High vol (> mean)')
ax.fill_between(df.index, df['volatility'], vol_mean, where=df['volatility'] < vol_mean,
                color='green', alpha=0.3, interpolate=True, label='Low vol (< mean)')
ax.set_title(f'{ticker} Rolling Volatility (20-day)')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized volatility')
ax.legend(loc='best')
ax.grid(True)

# bottom right: drawdown (layered transparent red fills for a gradient look)
ax = axes[1, 1]
for scale in [1.0, 0.75, 0.5, 0.25]:
    ax.fill_between(df.index, df['drawdown'] * scale, 0, color='red', alpha=0.2)
ax.plot(df.index, df['drawdown'], color='darkred', linewidth=1)
ax.scatter([dd_min_date], [dd_min_value], color='black', zorder=5, label=f'Max drawdown {dd_min_value:.1%}')
ax.set_title(f'{ticker} Maximum Drawdown')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown from running peak')
ax.legend(loc='best')
ax.grid(True)

fig.autofmt_xdate()
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Summary statistics

- Total return = last adjusted close / first adjusted close - 1
- Annualized return = exp(sum of log returns * 252 / n) - 1 (n = number of returns)
- Annualized volatility = std of log returns * square root of 252 (whole period; different from the mean rolling volatility used in Part 3)
- Best/worst day and up/down averages use log returns
- Positive days = positive returns divided by all returns (zero days included in the count)

In [ ]:
returns = df['log_return'].dropna()
n_returns = len(returns)

total_return = df['adj_close'].iloc[-1] / df['adj_close'].iloc[0] - 1
annualized_return = np.exp(returns.sum() * 252 / n_returns) - 1
annualized_vol = returns.std() * np.sqrt(252)
sharpe = calculate_sharpe_ratio(returns)
max_drawdown = df['drawdown'].min()
best_day = returns.max()
worst_day = returns.min()
positive_days = (returns > 0).sum() / n_returns
up_days = returns[returns > 0]
down_days = returns[returns < 0]
avg_up = up_days.mean() if len(up_days) > 0 else np.nan
avg_down = down_days.mean() if len(down_days) > 0 else np.nan


def show_percent(value, sign=True):
    """Format a fraction as a percent string, or N/A when it is not available."""
    if np.isnan(value):
        return 'N/A'
    if sign:
        return f'{value * 100:+.2f}%'
    return f'{value * 100:.2f}%'


summary_table = pd.DataFrame({
    'Metric': ['Total Return', 'Annualized Return', 'Annualized Volatility', 'Sharpe Ratio',
               'Maximum Drawdown', 'Best Day', 'Worst Day', 'Positive Days',
               'Average Up Day', 'Average Down Day'],
    'Value': [show_percent(total_return), show_percent(annualized_return),
              show_percent(annualized_vol, sign=False),
              'N/A' if np.isnan(sharpe) else f'{sharpe:.2f}',
              show_percent(max_drawdown), show_percent(best_day), show_percent(worst_day),
              show_percent(positive_days, sign=False), show_percent(avg_up), show_percent(avg_down)],
})
print(f'{ticker} summary ({df.index.min().date()} to {df.index.max().date()}, {len(df)} trading days)')
print(summary_table.to_string(index=False))

In [ ]:
sb.glue('ticker', ticker)
sb.glue('start_date', start_date)
sb.glue('end_date', end_date)
sb.glue('first_date', str(df.index.min().date()))
sb.glue('last_date', str(df.index.max().date()))
sb.glue('trading_days', int(len(df)))
sb.glue('avg_daily_return', float(returns.mean()))
sb.glue('avg_volatility', float(df['volatility'].mean()))
sb.glue('max_drawdown', float(max_drawdown))
sb.glue('sharpe_ratio', float(sharpe))
sb.glue('total_return', float(total_return))